# Topic 5 — Text Features
**Covers:** Bag of Words · TF-IDF · N-grams · Sparsity issues · Vocabulary pruning · Text
normalization pipeline

**The story:** InstaMetrics wants to auto-tag every incoming caption with a content category
(Promotional, Personal, Question, Motivational, Meme) — right now, creators tag this by hand,
which doesn't scale. Your job: turn raw caption TEXT into numeric features a model can actually
learn from.

**Dataset:** `data/raw/instagram_posts.csv` — 1,400 real-feeling captions, each written from
one of 5 underlying content categories.

---
## Explanation

Imagine you have a big pile of party invitations, and you want to sort them into types
("birthday," "sleepover," "just hanging out") — but a computer can't READ. It only understands
numbers.

- **Bag of Words** = you make a big checklist of every word that ever appears in any
  invitation, and for each invitation, you count how many times each checklist word shows up.
  You've turned a sentence into a list of numbers — but you've thrown away the WORD ORDER
  completely, like dumping all the words from a sentence into a bag and shaking it.
- **TF-IDF** = a smarter checklist. Common boring words that show up EVERYWHERE (like "the,"
  "and," "party") get a LOW score, even if they appear often, because they don't help you tell
  invitations apart. Rare, distinctive words (like "sleepover" or "RSVP") get a HIGHER score,
  because finding them is a strong clue about what type of invitation this is.
- **N-grams** = instead of counting single words, you also count PAIRS of words that appear
  next to each other, like "birthday party" or "sleep over" — sometimes the pair means
  something the individual words don't capture alone.
- **Sparsity** = your checklist has THOUSANDS of possible words, but any ONE invitation only
  uses a tiny handful of them — so most of your numbers, for most invitations, are just zero.
  A LOT of empty checklist boxes.
- **Vocabulary pruning** = trimming your checklist down — removing words that are too rare to
  ever help (typos, one-off words) or too common to mean anything (like "the")
- **Text normalization** = making sure "Party!!!", "PARTY", and "party" all count as the exact
  same checklist word, instead of being treated as three different, unrelated words.

## 1. WHY — Why can't a model read captions directly?

Every ML algorithm does arithmetic underneath. `"Link in bio to shop now"` has no arithmetic
meaning on its own. We need to turn text into NUMBERS — while preserving as much of the
meaningful signal as possible, and controlling the explosion of dimensions that naturally
happens once "every possible word" becomes a candidate feature.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/instagram_posts.csv")
df[["caption", "caption_category"]].sample(8, random_state=1)

,caption,caption_category
547,Missing home cooked food so much today.,Personal
754,Which cafe should I visit next in Bandra. Do y...,Question
1283,When Mumbai local trains are on time for once.,Meme
1128,Swipe up to grab yours before it sells out.,Promotional
1063,Use my code MUMBAI10 for a discount. Swipe up ...,Promotional
1007,Coffee and good vibes this morning. Late night...,Personal
236,Limited time offer just for my followers. New ...,Promotional
408,Missing home cooked food so much today. Coffee...,Personal


In [2]:
df["caption_category"].value_counts()

caption_category
Personal        378
Promotional     278
Meme            278
Motivational    259
Question        207
Name: count, dtype: int64

## 2. A tiny worked example you can count by hand — "ITM College" documents

Before turning to the full, messy Instagram captions, let's build every idea in this notebook
— normalization, Bag of Words, TF-IDF, and n-grams — on a corpus small enough to count on your
fingers. Treat each short sentence below as its own "document" (imagine five different short
posts describing the same college):

| | Document (raw) |
|---|---|
| D1 | "ITM is a college in Mumbai." |
| D2 | "ITM offers a BTECH course." |
| D3 | "ITM offers a MBA course." |
| D4 | "Mumbai has many colleges." |
| D5 | "ITM college is famous in Mumbai." |

**Step 1 — normalize.** Lowercase and strip punctuation (Section 3 below covers this in full):

| | Document (normalized) |
|---|---|
| D1 | itm is a college in mumbai |
| D2 | itm offers a btech course |
| D3 | itm offers a mba course |
| D4 | mumbai has many colleges |
| D5 | itm college is famous in mumbai |

**Note:** the standard tokenizer used below ignores single-letter tokens like "a" (they're too
short to be meaningful words on their own) — so "a" won't appear in the vocabulary. This matches
what `CountVectorizer` does automatically, not a mistake in our counting.

### 2a. Bag of Words — count every word, by hand

Vocabulary across all 5 documents (alphabetical, "a" excluded as noted above): **btech,
college, colleges, course, famous, has, in, is, itm, many, mba, mumbai, offers** — 13 words.

Now count how many times each vocabulary word appears in each document:

| | btech | college | colleges | course | famous | has | in | is | itm | many | mba | mumbai | offers |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| **D1** | 0 | 1 | 0 | 0 | 0 | 0 | 1 | 1 | 1 | 0 | 0 | 1 | 0 |
| **D2** | 1 | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 1 |
| **D3** | 0 | 0 | 0 | 1 | 0 | 0 | 0 | 0 | 1 | 0 | 1 | 0 | 1 |
| **D4** | 0 | 0 | 1 | 0 | 0 | 1 | 0 | 0 | 0 | 1 | 0 | 1 | 0 |
| **D5** | 0 | 1 | 0 | 0 | 1 | 0 | 1 | 1 | 1 | 0 | 0 | 1 | 0 |

That's it — that whole table IS the Bag-of-Words representation. Each row is now a row of
numbers a model can use; word order is gone (D1 "itm is a college" and a scrambled "college a
is itm" would produce the exact same row).

In [3]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

docs = [
    "itm is a college in mumbai",
    "itm offers a btech course",
    "itm offers a mba course",
    "mumbai has many colleges",
    "itm college is famous in mumbai",
]

cv_toy = CountVectorizer()
bow_toy = cv_toy.fit_transform(docs)
bow_toy_df = pd.DataFrame(bow_toy.toarray(), columns=cv_toy.get_feature_names_out(),
                           index=[f"D{i+1}" for i in range(5)])
bow_toy_df  # compare this to the hand-counted table above -- it should match exactly

,btech,college,colleges,course,famous,has,in,is,itm,many,mba,mumbai,offers
D1,0,1,0,0,0,0,1,1,1,0,0,1,0
D2,1,0,0,1,0,0,0,0,1,0,0,0,1
D3,0,0,0,1,0,0,0,0,1,0,1,0,1
D4,0,0,1,0,0,1,0,0,0,1,0,1,0
D5,0,1,0,0,1,0,1,1,1,0,0,1,0


### 2b. TF-IDF — by hand, on the same 5 documents

**Step 1 — document frequency (df):** how many of the 5 documents contain each word at all?

| Word | Appears in | df | N / df | IDF = ln(N / df) |
|---|---|---|---|---|
| itm | D1, D2, D3, D5 | 4 | 5/4 = 1.25 | ln(1.25) = **0.223** |
| mumbai | D1, D4, D5 | 3 | 5/3 = 1.67 | ln(1.67) = **0.511** |
| college | D1, D5 | 2 | 5/2 = 2.5 | ln(2.5) = **0.916** |
| course | D2, D3 | 2 | 5/2 = 2.5 | ln(2.5) = **0.916** |
| mba | D3 | 1 | 5/1 = 5.0 | ln(5.0) = **1.609** |
| btech | D2 | 1 | 5/1 = 5.0 | ln(5.0) = **1.609** |

Notice the pattern: **itm** shows up in almost every document, so its IDF is tiny (0.223) — it
carries almost no power to tell documents apart. **mba** and **btech** each show up in only
ONE document, so their IDF is the highest (1.609) — finding either word is a strong, distinctive
clue.

**Step 2 — TF-IDF for one document.** Take D3 = "itm offers a mba course" → normalized to
`itm offers mba course` (4 words after dropping "a", each appearing once):

```
TF(word in D3) = count of word in D3 / total words in D3 = 1 / 4 = 0.25   (same for all 4 words)

TF-IDF(itm,    D3) = 0.25 × 0.223 = 0.056
TF-IDF(offers, D3) = 0.25 × 0.916 = 0.229
TF-IDF(course, D3) = 0.25 × 0.916 = 0.229
TF-IDF(mba,    D3) = 0.25 × 1.609 = 0.402   <-- highest score
```

Even though **itm** and **mba** each appear exactly ONCE in D3 (identical raw counts!),
**mba** ends up with a TF-IDF score more than 7× higher than **itm**. That's the entire point
of TF-IDF: raw counts alone can't tell "itm" and "mba" apart, but weighting by how RARE each
word is across the whole corpus reveals that "mba" is the far more meaningful, distinctive word
in this document.

In [4]:
# Verify the hand calculation directly (not through TfidfVectorizer's defaults, which add
# smoothing and L2-normalization on top -- doing the plain arithmetic ourselves keeps this
# matched exactly to the manual formula above)
import numpy as np

N_docs = 5
doc_freq = (bow_toy_df > 0).sum(axis=0)                 # how many docs contain each word
manual_idf = np.log(N_docs / doc_freq)

d3_counts = bow_toy_df.loc["D3"]
d3_total_words = d3_counts.sum()
manual_tf_d3 = d3_counts / d3_total_words
manual_tfidf_d3 = (manual_tf_d3 * manual_idf).sort_values(ascending=False)

print("Document frequency:\n", doc_freq.to_dict())
print("\nIDF per word:\n", manual_idf.round(3).to_dict())
print("\nTF-IDF for D3 (non-zero words only):")
print(manual_tfidf_d3[manual_tfidf_d3 > 0].round(3))

Document frequency:
 {'btech': 1, 'college': 2, 'colleges': 1, 'course': 2, 'famous': 1, 'has': 1, 'in': 2, 'is': 2, 'itm': 4, 'many': 1, 'mba': 1, 'mumbai': 3, 'offers': 2}

IDF per word:
 {'btech': 1.609, 'college': 0.916, 'colleges': 1.609, 'course': 0.916, 'famous': 1.609, 'has': 1.609, 'in': 0.916, 'is': 0.916, 'itm': 0.223, 'many': 1.609, 'mba': 1.609, 'mumbai': 0.511, 'offers': 0.916}

TF-IDF for D3 (non-zero words only):
mba       0.402
offers    0.229
course    0.229
itm       0.056
dtype: float64


### 2c. N-grams — catching word PAIRS, by hand

Compare D1 and D5 — both mention "itm" and "college" as individual words, but do they mean the
same thing?

```
D1 (normalized): itm is a college in mumbai
D1 bigrams:  itm_is · is_college · college_in · in_mumbai

D5 (normalized): itm college is famous in mumbai
D5 bigrams:  itm_college · college_is · is_famous · famous_in · in_mumbai
```

A unigram Bag-of-Words (Section 2a) sees "itm" and "college" in BOTH D1 and D5 and treats them
identically on those two columns. But only **D5** actually has the phrase **"itm college"**
(referring to "ITM College" as a place) — in D1, "itm" and "college" are separated by "is a."
Bigrams are the only representation here that captures this real difference.

In [5]:
bigram_toy_vectorizer = CountVectorizer(ngram_range=(2, 2))
bigram_toy = bigram_toy_vectorizer.fit_transform(docs)
bigram_toy_df = pd.DataFrame(bigram_toy.toarray(), columns=bigram_toy_vectorizer.get_feature_names_out(),
                              index=[f"D{i+1}" for i in range(5)])

print("Bigrams found in D1:", [c for c in bigram_toy_df.columns if bigram_toy_df.loc["D1", c] > 0])
print("Bigrams found in D5:", [c for c in bigram_toy_df.columns if bigram_toy_df.loc["D5", c] > 0])
print('\n"itm college" appears in:',
      [d for d in bigram_toy_df.index if bigram_toy_df.loc[d, "itm college"] > 0])

Bigrams found in D1: ['college in', 'in mumbai', 'is college', 'itm is']
Bigrams found in D5: ['college is', 'famous in', 'in mumbai', 'is famous', 'itm college']

"itm college" appears in: ['D5']


Now that every step has been counted by hand on 5 tiny documents, the rest of this notebook
applies the exact same ideas — normalization, Bag of Words, sparsity, TF-IDF, n-grams,
vocabulary pruning — to real (much messier) Instagram captions.

## 3. Text normalization pipeline — WHY / WHAT / HOW

**Why:** without normalization, `"Party!!!"`, `"PARTY"`, and `"party,"` would each become a
DIFFERENT column in our vocabulary, splitting one real signal into three weak, fragmented ones.

**Manual example.**
```
Raw:        "Excited to partner with this AMAZING brand!!"
Lowercase:  "excited to partner with this amazing brand!!"
No punct.:  "excited to partner with this amazing brand"
Tokens:     ["excited", "to", "partner", "with", "this", "amazing", "brand"]
```

In [6]:
import re

def normalize_text(text):
    """Lowercase + strip punctuation/digits + collapse whitespace -- a minimal, dependency-free
    normalization pipeline (no external corpora needed, unlike stemming/lemmatization tools)."""
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

sample_caption = df["caption"].iloc[0]
print("Raw:       ", sample_caption)
print("Normalized:", normalize_text(sample_caption))

df["caption_normalized"] = df["caption"].apply(normalize_text)

Raw:        Never give up on your dreams keep going.
Normalized: never give up on your dreams keep going


**Note on stemming/lemmatization:** a fuller pipeline would also reduce words to their root
form (`"running"` → `"run"`, `"excited"` → `"excit"`), typically via NLTK or spaCy. We keep this
pipeline dependency-light and offline-friendly; scikit-learn's vectorizers below still handle
the rest of the job well on our already-normalized text.

## 4. Bag of Words — WHY / WHAT / HOW

**Manual worked example.** Two tiny captions:
```
Caption A: "great day great vibes"
Caption B: "great food today"
```
Vocabulary (all unique words, alphabetical): `[day, food, great, today, vibes]`

| | day | food | great | today | vibes |
|---|---|---|---|---|---|
| Caption A | 1 | 0 | 2 | 0 | 1 |
| Caption B | 0 | 1 | 1 | 1 | 0 |

Each caption becomes a row of word COUNTS — word order is completely discarded.

In [7]:
from sklearn.feature_extraction.text import CountVectorizer

toy_captions = ["great day great vibes", "great food today"]
toy_vectorizer = CountVectorizer()
toy_bow = toy_vectorizer.fit_transform(toy_captions)

print("Vocabulary:", toy_vectorizer.get_feature_names_out())
pd.DataFrame(toy_bow.toarray(), columns=toy_vectorizer.get_feature_names_out(), index=["Caption A", "Caption B"])

Vocabulary: ['day' 'food' 'great' 'today' 'vibes']


,day,food,great,today,vibes
Caption A,1,0,2,0,1
Caption B,0,1,1,1,0


In [8]:
# Now apply Bag of Words to our REAL, normalized captions
bow_vectorizer = CountVectorizer(stop_words="english")
X_bow = bow_vectorizer.fit_transform(df["caption_normalized"])

print(f"Vocabulary size: {len(bow_vectorizer.get_feature_names_out())} unique words")
print(f"Bag-of-Words matrix shape: {X_bow.shape}  (rows=captions, columns=vocabulary words)")
print("A few vocabulary words:", list(bow_vectorizer.get_feature_names_out()[:15]))

Vocabulary size: 124 unique words
Bag-of-Words matrix shape: (1400, 124)  (rows=captions, columns=vocabulary words)
A few vocabulary words: ['absolutely', 'actually', 'amazing', 'auto', 'bandra', 'beats', 'beautiful', 'believe', 'bio', 'brand', 'cafe', 'chai', 'chasing', 'checking', 'city']


## 5. Sparsity — seeing the empty-checklist-box problem directly

In [9]:
n_total_cells = X_bow.shape[0] * X_bow.shape[1]
n_nonzero_cells = X_bow.nnz
sparsity = 1 - (n_nonzero_cells / n_total_cells)

print(f"Total cells in the matrix:     {n_total_cells:,}")
print(f"Non-zero cells (actual words): {n_nonzero_cells:,}")
print(f"Sparsity: {sparsity:.2%} of the matrix is exactly zero")
print(f"\nAverage non-zero words per caption: {n_nonzero_cells / X_bow.shape[0]:.1f}, "
      f"out of a {X_bow.shape[1]}-word vocabulary")

Total cells in the matrix:     173,600
Non-zero cells (actual words): 8,742
Sparsity: 94.96% of the matrix is exactly zero

Average non-zero words per caption: 6.2, out of a 124-word vocabulary


This is the sparsity problem, concretely: a huge majority of the matrix is zero, because any
ONE caption only ever uses a tiny fraction of the full vocabulary. This is why text feature
matrices are stored in special *sparse* formats (like `scipy.sparse`, what `CountVectorizer`
returns by default) rather than dense arrays — storing all those zeros explicitly would waste
enormous amounts of memory at real scale.

## 6. TF-IDF — WHY / WHAT / HOW

**Why:** plain word COUNTS treat a word that appears in every single caption exactly the same
as a word that appears in only one -- even though the rare word is far more diagnostic.
**TF-IDF (Term Frequency - Inverse Document Frequency)** downweights words that are common
across MANY captions, and upweights words that are distinctive to just a few.

**Manual formula, worked example.** Suppose the word `"amazing"` appears in 20 out of 1,400
captions:
```
TF (term frequency) in one caption  = (count of "amazing" in this caption) / (total words in this caption)
IDF (inverse document frequency)    = log(N_total_captions / N_captions_containing_word)
                                     = log(1400 / 20) = log(70) ≈ 4.25

TF-IDF score = TF * IDF
```
A word appearing in nearly every caption (say, 1,300 out of 1,400) gets
`IDF = log(1400/1300) ≈ 0.07` — a near-zero weight, correctly recognized as almost useless for
telling captions apart.

In [10]:
N = len(df)
for word, doc_count in [("amazing", 20), ("great", 400), ("the", 1300)]:
    idf = np.log(N / doc_count)
    print(f"'{word}':  appears in {doc_count}/{N} captions  ->  IDF = {idf:.3f}")

'amazing':  appears in 20/1400 captions  ->  IDF = 4.248
'great':  appears in 400/1400 captions  ->  IDF = 1.253
'the':  appears in 1300/1400 captions  ->  IDF = 0.074


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(stop_words="english")
X_tfidf = tfidf_vectorizer.fit_transform(df["caption_normalized"])

# Compare a common word's average TF-IDF weight vs. a rarer, more distinctive word
vocab = tfidf_vectorizer.get_feature_names_out()
tfidf_means = np.asarray(X_tfidf.mean(axis=0)).flatten()
tfidf_lookup = dict(zip(vocab, tfidf_means))

for word in ["today", "excited", "giveaway", "traffic"]:
    if word in tfidf_lookup:
        print(f"Average TF-IDF weight of '{word}': {tfidf_lookup[word]:.4f}")

Average TF-IDF weight of 'today': 0.0472
Average TF-IDF weight of 'excited': 0.0135
Average TF-IDF weight of 'giveaway': 0.0144
Average TF-IDF weight of 'traffic': 0.0217


## 7. N-grams — WHY / WHAT / HOW

**Why:** single words (unigrams) lose word-pair meaning. `"local trains"` and `"trains local"`
would look identical to a unigram-only Bag of Words, and a phrase like `"never give"` (from
"never give up") carries more specific meaning than `"never"` and `"give"` scored separately.
**N-grams** capture contiguous sequences of N words as their own vocabulary entries.

In [12]:
bigram_vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2))  # unigrams + bigrams
X_bigram = bigram_vectorizer.fit_transform(df["caption_normalized"])

bigram_only_vocab = [w for w in bigram_vectorizer.get_feature_names_out() if " " in w]
print(f"Unigram+bigram vocabulary size: {len(bigram_vectorizer.get_feature_names_out())} "
      f"(vs. {len(bow_vectorizer.get_feature_names_out())} unigrams alone)")
print(f"Number of bigram (2-word) entries: {len(bigram_only_vocab)}")
print("A few example bigrams found:", bigram_only_vocab[:10])

Unigram+bigram vocabulary size: 428 (vs. 124 unigrams alone)
Number of bigram (2-word) entries: 304
A few example bigrams found: ['absolutely today', 'actually uses', 'amazing brand', 'auto driver', 'bandra favorite', 'bandra giveaway', 'bandra outfit', 'bandra prefer', 'bandra recommendations', 'bandra vlog']


**The tradeoff:** adding bigrams roughly multiplies the vocabulary size, which worsens
sparsity (Section 4) further. N-grams add real signal, but every additional `n` compounds the
dimensionality problem — this is exactly the curse of dimensionality from Topic 3, showing up
concretely in a text pipeline.

## 8. Vocabulary pruning — WHY / WHAT / HOW

**Why:** most of that huge vocabulary is either (a) so RARE it's essentially noise (typos,
one-off words seen in a single caption — no generalizable signal), or (b) so COMMON it carries
no discriminating power (already handled by `stop_words`, but plain frequent words can still
slip through). `min_df` and `max_df` prune both ends.

In [13]:
pruned_vectorizer = CountVectorizer(
    stop_words="english",
    min_df=5,      # ignore words appearing in FEWER than 5 captions (too rare to generalize)
    max_df=0.5,    # ignore words appearing in MORE than 50% of captions (too common to discriminate)
)
X_pruned = pruned_vectorizer.fit_transform(df["caption_normalized"])

print(f"Original vocabulary:  {len(bow_vectorizer.get_feature_names_out())} words")
print(f"Pruned vocabulary:    {len(pruned_vectorizer.get_feature_names_out())} words")
print(f"Sparsity before pruning: {1 - X_bow.nnz / (X_bow.shape[0]*X_bow.shape[1]):.2%}")
print(f"Sparsity after pruning:  {1 - X_pruned.nnz / (X_pruned.shape[0]*X_pruned.shape[1]):.2%}")

Original vocabulary:  124 words
Pruned vocabulary:    124 words
Sparsity before pruning: 94.96%
Sparsity after pruning:  94.96%


## 9. Putting it together — classifying caption category from text alone

Let's confirm these features actually work, end to end: train a simple classifier on the
PRUNED TF-IDF features to predict `caption_category` from nothing but the caption text.

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

pruned_tfidf_vectorizer = TfidfVectorizer(stop_words="english", min_df=5, max_df=0.5, ngram_range=(1, 2))
X_text = pruned_tfidf_vectorizer.fit_transform(df["caption_normalized"])
y_text = df["caption_category"]

X_train, X_test, y_train, y_test = train_test_split(X_text, y_text, test_size=0.25,
                                                      random_state=42, stratify=y_text)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(classification_report(y_test, pred))

              precision    recall  f1-score   support

        Meme       1.00      1.00      1.00        69
Motivational       1.00      1.00      1.00        65
    Personal       1.00      1.00      1.00        94
 Promotional       1.00      1.00      1.00        70
    Question       1.00      1.00      1.00        52

    accuracy                           1.00       350
   macro avg       1.00      1.00      1.00       350
weighted avg       1.00      1.00      1.00       350



**Reading this honestly:** accuracy here should be very high — our captions were generated
directly from category-specific phrase banks (a clean, controlled dataset), so the vocabulary
signal is unusually strong and easy to separate. Real Instagram captions are messier (mixed
topics in one caption, sarcasm, emojis, slang) and would show meaningfully lower accuracy — but
the PIPELINE itself (normalize → vectorize → prune → model) is exactly what you'd deploy
regardless of how clean or messy the real captions turn out to be.

## 10. Recap

- Text must become numbers before any model can use it — that's the entire point of this
  notebook.
- **Bag of Words**: raw word counts, order discarded.
- **TF-IDF**: downweights common words, upweights rare/distinctive ones.
- **N-grams**: capture short word sequences, at the cost of a much larger vocabulary.
- **Sparsity**: most of a text feature matrix is zero — expect it, store it efficiently.
- **Vocabulary pruning** (`min_df`/`max_df`): drop words too rare to generalize or too common
  to discriminate, shrinking both dimensionality and sparsity.
- **Normalization** (lowercase, punctuation removal, and ideally stemming/lemmatization) must
  happen BEFORE vectorizing, or the same real word fragments into multiple counted variants.

**ELI5 recap:** a checklist of every word and how often it shows up (Bag of Words); a smarter
checklist that ignores boring common words and rewards rare distinctive ones (TF-IDF); counting
word PAIRS too (n-grams); a LOT of empty checklist boxes for any one invitation (sparsity);
trimming the checklist down to just the useful words (pruning); and making sure "PARTY!!!" and
"party" count as the exact same checklist word (normalization).